# Baseline tests



In [1]:
import time
import json

llm

In [2]:
import torch

GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT >= 2:
    LLM_DEVICE_MAP = "balanced_low_0"
    PPO_DEVICE = "cuda:0"
    print(f"{GPU_COUNT} GPU CUDA detectes: LLM reparti sur les GPU disponibles.")
elif GPU_COUNT == 1:
    LLM_DEVICE_MAP = "auto"
    PPO_DEVICE = "cuda:0"
    print("Un seul GPU CUDA detecte: execution sur cuda:0.")
else:
    LLM_DEVICE_MAP = "cpu"
    PPO_DEVICE = "cpu"
    print("Aucun GPU detecte: execution CPU pour validation locale.")

EVAL_BATCH_SIZE = 8

2 GPU CUDA detectes: LLM reparti sur les GPU disponibles.


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

import torch


class FrozenLLM:
    """
    Wrapper around a frozen causal language model for prompt optimization.

    The model is loaded in inference mode and its parameters are never
    updated. The class is responsible only for loading the model and
    generating responses.

    Parameters
    ----------
    model_name : str, default="Qwen/Qwen2.5-3B-Instruct"
        Hugging Face model identifier.

    max_new_tokens : int, default=512
        Maximum number of tokens generated for each response.

    temperature : float, default=0.0
        Sampling temperature. A value of 0 uses deterministic generation.

    device_map : str, default="auto"
        Device mapping used to load the model.
    """

    def __init__(
        self,
        model_name="Qwen/Qwen2.5-3B-Instruct",
        max_new_tokens=512,
        temperature=0.0,
        device_map="auto",
    ):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

        # ---------------------------------------------------------
        # Tokenizer
        # ---------------------------------------------------------

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name
        )
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token


        # Qwen may not have a pad token explicitly defined.
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = (
                self.tokenizer.eos_token
            )

        # ---------------------------------------------------------
        # Model
        # ---------------------------------------------------------

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map=device_map,
        )

        # ---------------------------------------------------------
        # Freeze model
        # ---------------------------------------------------------

        self.model.eval()

        for parameter in self.model.parameters():
            parameter.requires_grad = False

    # =============================================================
    # GENERATION CONFIGURATION
    # =============================================================

    def _generation_kwargs(self):
        """
        Build generation parameters.

        Returns
        -------
        dict
            Parameters passed to ``model.generate``.
        """

        generation_kwargs = {
            "max_new_tokens": self.max_new_tokens,
            "pad_token_id": self.tokenizer.pad_token_id,
        }

        if self.temperature > 0:
            generation_kwargs.update({
                "do_sample": True,
                "temperature": self.temperature,
            })
        else:
            generation_kwargs.update({
                "do_sample": False,
            })

        return generation_kwargs

    # =============================================================
    # SINGLE GENERATION
    # =============================================================

    def generate(self, prompt):
        """
        Generate a response for a single prompt.

        Parameters
        ----------
        prompt : str
            Input prompt.

        Returns
        -------
        str
            Generated response.
        """

        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]

        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(self.model.device)
            for key, value in inputs.items()
        }

        with torch.inference_mode():

            outputs = self.model.generate(
                **inputs,
                **self._generation_kwargs(),
            )

        input_length = inputs["input_ids"].shape[-1]

        generated_tokens = outputs[0][
            input_length:
        ]

        response = self.tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        )

        return response.strip()

    # =============================================================
    # BATCH GENERATION
    # =============================================================

    def generate_batch(self, prompts):
        """
        Generate responses for multiple prompts simultaneously.

        Parameters
        ----------
        prompts : list[str]
            List of input prompts.

        Returns
        -------
        list[str]
            Generated responses in the same order as the input prompts.
        """

        if not prompts:
            return []

        messages = [
            [
                {
                    "role": "user",
                    "content": prompt,
                }
            ]
            for prompt in prompts
        ]

        # ---------------------------------------------------------
        # Tokenization
        # ---------------------------------------------------------

        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        )

        inputs = {
            key: value.to(self.model.device)
            for key, value in inputs.items()
        }

        # Length of the padded input sequence.
        input_length = inputs["input_ids"].shape[1]

        # ---------------------------------------------------------
        # Generation
        # ---------------------------------------------------------

        with torch.inference_mode():

            outputs = self.model.generate(
                **inputs,
                **self._generation_kwargs(),
            )

        # ---------------------------------------------------------
        # Extract generated tokens
        # ---------------------------------------------------------

        responses = []

        for output in outputs:

            generated_tokens = output[input_length:]

            response = self.tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True,
            )

            responses.append(
                response.strip()
            )

        return responses

evaluator

In [4]:
import re


class GSM8KEvaluator:
    """
    Evaluate a frozen LLM on GSM8K problems.

    The evaluator supports:

    - prompt-level caching;
    - batched LLM generation;
    - accuracy computation;
    - evaluation statistics.

    Parameters
    ----------
    llm : object
        Frozen language model exposing ``generate(prompt)``
        and ``generate_batch(prompts)`` methods.

    dataset : iterable
        GSM8K dataset containing ``question`` and ``answer`` fields.

    batch_size : int, default=8
        Number of GSM8K problems evaluated simultaneously.
    """

    def __init__(
        self,
        llm,
        dataset,
        batch_size=8,
    ):
        self.llm = llm
        self.dataset = dataset
        self.batch_size = batch_size

        # ---------------------------------------------------------
        # Prompt-level cache
        # ---------------------------------------------------------

        self.cache = {}

        # ---------------------------------------------------------
        # Evaluation statistics
        # ---------------------------------------------------------

        self.cache_hits = 0
        self.cache_misses = 0
        self.llm_calls = 0

    # =============================================================
    # EVALUATION
    # =============================================================

    def evaluate(
        self,
        prompt,
        return_details=False,
    ):
        """
        Evaluate a prompt on the GSM8K dataset.

        Parameters
        ----------
        prompt : str
            Prompt instruction to evaluate.

        return_details : bool, default=False
            If True, return detailed results for each problem.

        Returns
        -------
        float or dict
            Accuracy if ``return_details=False``.

            Otherwise, a dictionary containing accuracy,
            individual results and evaluation statistics.
        """

        # ---------------------------------------------------------
        # Prompt-level cache
        # ---------------------------------------------------------

        if prompt in self.cache:

            self.cache_hits += 1

            cached_result = self.cache[prompt]

            if return_details:
                return cached_result

            return cached_result["accuracy"]

        # ---------------------------------------------------------
        # Cache miss
        # ---------------------------------------------------------

        self.cache_misses += 1

        # ---------------------------------------------------------
        # Build all GSM8K prompts
        # ---------------------------------------------------------

        examples = list(self.dataset)

        prompts = [
            self.build_prompt(
                prompt,
                example["question"],
            )
            for example in examples
        ]

        # ---------------------------------------------------------
        # Batched generation
        # ---------------------------------------------------------

        responses = []

        for start in range(
            0,
            len(prompts),
            self.batch_size,
        ):

            batch_prompts = prompts[
                start:start + self.batch_size
            ]

            batch_responses = (
                self.llm.generate_batch(
                    batch_prompts
                )
            )

            responses.extend(
                batch_responses
            )

            self.llm_calls += len(
                batch_prompts
            )

        # ---------------------------------------------------------
        # Evaluate predictions
        # ---------------------------------------------------------

        results = []

        for example, response in zip(
            examples,
            responses,
        ):

            question = example["question"]

            target = self.extract_target_answer(
                example["answer"]
            )

            prediction = self.extract_prediction(
                response
            )

            correct = self.is_correct(
                prediction,
                target,
            )

            results.append({
                "question": question,
                "target": target,
                "response": response,
                "prediction": prediction,
                "correct": correct,
            })

        # ---------------------------------------------------------
        # Accuracy
        # ---------------------------------------------------------

        accuracy = (
            sum(
                result["correct"]
                for result in results
            )
            / len(results)
        )

        # ---------------------------------------------------------
        # Cache complete result
        # ---------------------------------------------------------

        evaluation_result = {
            "accuracy": accuracy,
            "results": results,
        }

        self.cache[prompt] = evaluation_result

        if return_details:
            return evaluation_result

        return accuracy

    # =============================================================
    # STATISTICS
    # =============================================================

    def get_statistics(self):
        """
        Return evaluator statistics.

        Returns
        -------
        dict
            Evaluation and cache statistics.
        """

        total_evaluations = (
            self.cache_hits
            + self.cache_misses
        )

        return {
            "total_evaluations": total_evaluations,
            "cache_hits": self.cache_hits,
            "cache_misses": self.cache_misses,
            "llm_calls": self.llm_calls,
            "cache_hit_rate": (
                self.cache_hits
                / total_evaluations
                if total_evaluations > 0
                else 0.0
            ),
        }

    # =============================================================
    # PROMPT BUILDING
    # =============================================================

    @staticmethod
    def build_prompt(
        prompt,
        question,
    ):
        """
        Combine the current instruction with a GSM8K question.

        Parameters
        ----------
        prompt : str
            Current optimized instruction.

        question : str
            GSM8K problem.

        Returns
        -------
        str
            Complete prompt sent to the LLM.
        """

        return f"""{prompt}

Problem:

{question}
"""

    # =============================================================
    # TARGET EXTRACTION
    # =============================================================

    @staticmethod
    def extract_target_answer(answer):
        """
        Extract the final numerical answer from a GSM8K target.

        Parameters
        ----------
        answer : str
            Original GSM8K answer containing reasoning and
            final answer.

        Returns
        -------
        str
            Normalized final answer.
        """

        match = re.search(
            r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)",
            answer,
        )

        if match is None:

            raise ValueError(
                f"Could not extract GSM8K target answer: "
                f"{answer}"
            )

        return GSM8KEvaluator.normalize_number(
            match.group(1)
        )

    # =============================================================
    # PREDICTION EXTRACTION
    # =============================================================

    @staticmethod
    def extract_prediction(response):
        """
        Extract a numerical prediction from an LLM response.

        Parameters
        ----------
        response : str
            Generated model response.

        Returns
        -------
        str or None
            Extracted numerical answer.
        """

        patterns = [
            r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)",

            r"(?:final answer|answer)"
            r"\s*(?:is|:)?\s*"
            r"([-+]?\d[\d,]*(?:\.\d+)?)",
        ]

        for pattern in patterns:

            matches = re.findall(
                pattern,
                response,
                flags=re.IGNORECASE,
            )

            if matches:

                return GSM8KEvaluator.normalize_number(
                    matches[-1]
                )

        # ---------------------------------------------------------
        # Fallback: last number
        # ---------------------------------------------------------

        numbers = re.findall(
            r"[-+]?\d[\d,]*(?:\.\d+)?",
            response,
        )

        if not numbers:
            return None

        return GSM8KEvaluator.normalize_number(
            numbers[-1]
        )

    # =============================================================
    # NUMBER NORMALIZATION
    # =============================================================

    @staticmethod
    def normalize_number(value):
        """
        Normalize a numerical answer.

        Parameters
        ----------
        value : str
            Numerical value.

        Returns
        -------
        str
            Normalized numerical representation.
        """

        value = value.strip()
        value = value.replace(",", "")

        try:

            number = float(value)

            if number.is_integer():
                return str(int(number))

            return str(number)

        except ValueError:

            return value

    # =============================================================
    # CORRECTNESS
    # =============================================================

    @staticmethod
    def is_correct(
        prediction,
        target,
    ):
        """
        Compare a prediction with the GSM8K target.

        Parameters
        ----------
        prediction : str or None
            Extracted model prediction.

        target : str
            Expected answer.

        Returns
        -------
        bool
            Whether the prediction is correct.
        """

        if prediction is None:
            return False

        return prediction == target

prompts

In [5]:
BASE_PROMPT = """Solve the following math problem.
Provide the final answer clearly."""

PROMPT_TRANSFORMATIONS = {

    0: {
        "name": "step_by_step",
        "instruction":"Solve the problem step by step."},
    1: {
        "name":"reasoning",
        "instruction":"Explain your reasoning clearly."},
    2: {
        "name":"verification",
        "instruction":"Verify your answer before giving the final answer."},
    3: {
        "name":"calculation_check",
        "instruction":"Check your calculations carefully."},
    4: {
        "name":"decomposition",
        "instruction":"Break the problem into smaller steps."},
    5: {
        "name":"relevant_information",
        "instruction":"Identify the relevant information before solving the problem."},
    6: {
        "name":"double_check",
        "instruction":"Double-check your final answer."},
    7: {
        "name":"answer_format",
        "instruction":"Clearly state the final answer at the end."
    }

}


def apply_transformation(prompt, action):
    """
    Apply a prompt transformation.
    args
    -------
    prompt : str
        current prompt.
    action : int
        ID of the transformation to apply
    Returns
    -------
    str
        Transformed prompt
    """
    transformation = PROMPT_TRANSFORMATIONS[action]["instruction"]

    if not transformation:
        return prompt
    return f"{prompt}\n\n{transformation}"

environment

In [6]:
import gymnasium as gym
import numpy as np


class PromptOptimizationEnv(gym.Env):
    """
    Gymnasium environment for prompt optimization.

    The agent sequentially selects prompt transformations.
    The LLM remains frozen and is evaluated after each transformation.

    Parameters
    ----------
    evaluator : GSM8KEvaluator
        Evaluator used to measure prompt performance.

    base_prompt : str
        Initial prompt before any transformation.

    max_steps : int, default=5
        Maximum number of transformations per episode.

    final_reward_coef : float, default=0.5
        Weight applied to the final improvement relative to
        the base prompt.
    """

    def __init__(
        self,
        evaluator,
        base_prompt,
        max_steps=5,
        final_reward_coef=0.5,
    ):
        super().__init__()

        self.evaluator = evaluator
        self.base_prompt = base_prompt
        self.max_steps = max_steps
        self.final_reward_coef = final_reward_coef

        # Number of available transformations
        self.action_dim = 8

        # ---------------------------------------------------------
        # Action space
        # ---------------------------------------------------------

        self.action_space = gym.spaces.Discrete(
            self.action_dim
        )

        # ---------------------------------------------------------
        # Observation space
        # ---------------------------------------------------------

        # Current accuracy
        # Previous accuracy
        # Step progress
        # Number of transformations already selected
        #
        # Plus one binary feature per action indicating whether
        # the transformation has already been selected.
        #
        # Total:
        # 4 + 8 = 12
        # ---------------------------------------------------------

        self.observation_dim = 12

        self.observation_space = gym.spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.observation_dim,),
            dtype=np.float32,
        )

        # ---------------------------------------------------------
        # Episode state
        # ---------------------------------------------------------

        self.current_prompt = None
        self.current_accuracy = 0.0
        self.base_accuracy = 0.0
        self.previous_accuracy = 0.0

        self.step_count = 0

        self.selected_actions = []

        self.used_actions = set()

    # =============================================================
    # OBSERVATION
    # =============================================================

    def _get_observation(self):
        """
        Build the current environment observation.

        Returns
        -------
        np.ndarray
            Current environment state.
        """

        step_progress = (
            self.step_count / self.max_steps
        )

        num_selected = (
            len(self.selected_actions)
            / self.action_dim
        )

        used_actions = np.zeros(
            self.action_dim,
            dtype=np.float32,
        )

        for action in self.used_actions:
            used_actions[action] = 1.0

        observation = np.concatenate(
            [
                np.array(
                    [
                        self.current_accuracy,
                        self.previous_accuracy,
                        step_progress,
                        num_selected,
                    ],
                    dtype=np.float32,
                ),
                used_actions,
            ]
        )

        return observation.astype(
            np.float32
        )

    # =============================================================
    # ACTION MASK
    # =============================================================

    def get_action_mask(self):
        """
        Return the mask of currently available actions.

        Returns
        -------
        np.ndarray
            Boolean mask where True means that the action
            can still be selected.
        """

        mask = np.ones(
            self.action_dim,
            dtype=bool,
        )

        for action in self.used_actions:
            mask[action] = False

        return mask

    # =============================================================
    # RESET
    # =============================================================

    def reset(
        self,
        *,
        seed=None,
        options=None,
    ):
        """
        Reset the environment.

        Returns
        -------
        observation : np.ndarray
            Initial observation.

        info : dict
            Initial environment information.
        """

        super().reset(seed=seed)

        self.current_prompt = (
            self.base_prompt
        )

        # ---------------------------------------------------------
        # Evaluate base prompt
        # ---------------------------------------------------------

        self.base_accuracy = (
            self.evaluator.evaluate(
                self.base_prompt
            )
        )

        self.current_accuracy = (
            self.base_accuracy
        )

        self.previous_accuracy = (
            self.base_accuracy
        )

        self.step_count = 0

        self.selected_actions = []

        self.used_actions = set()

        observation = (
            self._get_observation()
        )

        info = {
            "prompt": self.current_prompt,
            "accuracy": self.current_accuracy,
            "base_accuracy": self.base_accuracy,
            "actions": self.selected_actions,
        }

        return observation, info

    # =============================================================
    # STEP
    # =============================================================

    def step(self, action):
        """
        Apply a prompt transformation.

        Parameters
        ----------
        action : int
            Transformation ID.

        Returns
        -------
        observation : np.ndarray
            New environment state.

        reward : float
            Reward obtained after the transformation.

        terminated : bool
            Whether the episode naturally ended.

        truncated : bool
            Whether the episode was truncated.

        info : dict
            Environment information.
        """

        # ---------------------------------------------------------
        # Validate action
        # ---------------------------------------------------------

        if action in self.used_actions:
            raise ValueError(
                f"Action {action} has already been selected."
            )

        if not self.action_space.contains(action):
            raise ValueError(
                f"Invalid action: {action}"
            )

        # ---------------------------------------------------------
        # Previous state
        # ---------------------------------------------------------

        self.previous_accuracy = (
            self.current_accuracy
        )

        # ---------------------------------------------------------
        # Apply transformation
        # ---------------------------------------------------------

        self.current_prompt = (
            apply_transformation(
                self.current_prompt,
                action,
            )
        )

        self.used_actions.add(action)

        self.selected_actions.append(
            action
        )

        self.step_count += 1

        # ---------------------------------------------------------
        # Evaluate transformed prompt
        # ---------------------------------------------------------

        self.current_accuracy = (
            self.evaluator.evaluate(
                self.current_prompt
            )
        )

        # ---------------------------------------------------------
        # Local reward
        # ---------------------------------------------------------

        reward = (
            self.current_accuracy
            - self.previous_accuracy
        )

        # ---------------------------------------------------------
        # Episode termination
        # ---------------------------------------------------------

        terminated = (
            self.step_count
            >= self.max_steps
        )

        truncated = False

        # ---------------------------------------------------------
        # Final reward
        # ---------------------------------------------------------

        final_reward = 0.0

        if terminated:

            final_improvement = (
                self.current_accuracy
                - self.base_accuracy
            )

            final_reward = (
                self.final_reward_coef
                * final_improvement
            )

            reward += final_reward

        # ---------------------------------------------------------
        # Observation
        # ---------------------------------------------------------

        observation = (
            self._get_observation()
        )

        # ---------------------------------------------------------
        # Information
        # ---------------------------------------------------------

        info = {
            "prompt": self.current_prompt,
            "accuracy": self.current_accuracy,
            "base_accuracy": self.base_accuracy,
            "improvement": (
                self.current_accuracy
                - self.base_accuracy
            ),
            "previous_accuracy": (
                self.previous_accuracy
            ),
            "local_reward": (
                self.current_accuracy
                - self.previous_accuracy
            ),
            "final_reward": final_reward,
            "actions": list(
                self.selected_actions
            ),
        }

        return (
            observation,
            reward,
            terminated,
            truncated,
            info,
        )

manual Baseline

In [7]:
class ManualBaseline:
    """
    Baseline based on manually designed prompt transformation sequences.

    Parameters
    ----------
    env : PromptOptimizationEnv
        Prompt optimization environment.

    action_sequences : list[list[int]]
        List of manually designed transformation sequences.
    """

    def __init__(self, env, action_sequences):
        self.env = env
        self.action_sequences = action_sequences

    def run(self):
        """
        Evaluate all manually designed prompt configurations.

        Returns
        -------
        dict
            Best manually designed prompt and results for all configurations.
        """

        results = []

        for actions in self.action_sequences:

            self.env.reset()

            for action in actions:

                _, _, terminated, truncated, info = (
                    self.env.step(action)
                )

                if terminated or truncated:
                    break

            results.append({
                "actions": actions.copy(),
                "prompt": info["prompt"],
                "accuracy": info["accuracy"],
            })

        best_result = max(
            results,
            key=lambda x: x["accuracy"]
        )

        return {
            "best_prompt": best_result["prompt"],
            "best_accuracy": best_result["accuracy"],
            "best_actions": best_result["actions"],
            "results": results,
        }

random_search

In [8]:
import random

class RandomSearch:
    """
    Random search baseline for prompt optimization.
    
    The algorithm randomly selects prompt transformations and evaluates
    the resulting prompt. The best-performing prompt found during the 
    search is returned

    Parameters
    ----------
    env : PromptOptimizationEnv
        Prompt optimization environment

    n_trials : int, default=20
        Number of random prompt configuration to evaluate.
    
    max_steps : int, default=3
        Maximum number of transformations in a single trial.
    
    seed : int or None, default=None
        Random seed for reproducibility.
    """ 

    def __init__(
            self,
            env,
            n_trials=20,
            max_steps=3,
            seed=None
    ):
        self.env=env
        self.n_trials=n_trials
        self.max_steps=max_steps
        if seed is not None:
            random.seed(seed)

    def run(self):
        """
        Execute the random search
        Returns
        -------
        dict
            Best prompt and associated information.
        """

        best_prompt = None
        best_accuracy = float("-inf")
        best_action= None

        trials_results = []
        for trial in range(self.n_trials):

            observation, info = self.env.reset()
            available_actions = list(
                range(self.env.action_dim)
                )
            selected_actions = []

            # random number of transformations
            n_steps = random.randint(
                1,
                self.max_steps,
            )

            for _ in range(n_steps):
                # select only transformation 
                # that have not already been used
                remaining_actions = [
                    action for action in available_actions
                    if action not in selected_actions
                ]

                if not remaining_actions:
                    break

                action = random.choice(
                    remaining_actions
                )
                selected_actions.append(action)
                observation, reward, terminated, truncated, info = (
                    self.env.step(action)
                )

                if terminated or truncated:
                    break

            accuracy = info['accuracy']
            result= {
                "trial": trial,
                "prompt": info["prompt"],
                "accuracy": accuracy,
                "actions": selected_actions.copy(),
            }

            trials_results.append(result)

            if accuracy > best_accuracy:
                best_accuracy=accuracy
                best_prompt=info["prompt"]
                best_action=selected_actions.copy()
        return {
                "best_prompt": best_prompt,
                "best_accuracy": best_accuracy,
                "best_action": best_action,
                "trials": trials_results
            }
    



In [9]:
from datasets import load_dataset

DATASET_SIZE = 50
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_STEPS = 5
RANDOM_SEARCH_TRIALS = 20
RANDOM_SEED = 42


dataset = load_dataset(
    "openai/gsm8k",
    "main"
)

train_dataset = dataset["train"].select(
    range(DATASET_SIZE)
)

llm = FrozenLLM(
    max_new_tokens=256,
    temperature=0.0,
    device_map=LLM_DEVICE_MAP,
    model_name=MODEL_NAME,
)

evaluator = GSM8KEvaluator(
    llm,
    train_dataset,
)

env = PromptOptimizationEnv(
    base_prompt=BASE_PROMPT,
    evaluator=evaluator,
    max_steps=MAX_STEPS
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

## Manual Baseline

In [12]:
manual_sequences = [
    [0],
    [3],
    [7],
    [0, 3],
    [3, 7],
    [0, 3, 7],
    [3, 5, 7],
    [0, 3, 5, 7],
    [0, 7, 2, 6],
    [0, 3, 7, 2, 6],
]

In [13]:
base_accuracy = evaluator.evaluate(
    BASE_PROMPT
)

manual_baseline = ManualBaseline(
    env=env,
    action_sequences=manual_sequences
)

start_time = time.time()
manual_result = manual_baseline.run()
manual_time = time.time() - start_time
print(
    f"Best accuracy :"
    f"{manual_result["best_accuracy"]: .4f}"
)

print(
    f"Improvement :"
    f"{manual_result["best_accuracy"] - base_accuracy}"
)

print(
    f"Best action :"
    f"{manual_result["best_actions"]}"
)

print(
    f"Time"
    f"{manual_time: .2f}"
)


Best accuracy : 0.4400
Improvement :0.06
Best action :[3, 7]
Time 1297.15


## Random search Baseline

In [14]:
random_baseline = RandomSearch(
    env=env,
    n_trials=RANDOM_SEARCH_TRIALS,
    max_steps=MAX_STEPS,
    seed = RANDOM_SEED
)

start_time = time.time()

random_result = random_baseline.run()
random_time = time.time() - start_time

print(
    f"Best accuracy :"
    f"{random_result["best_accuracy"]: .4f}"
)

print(
    f"Improvement :"
    f"{random_result["best_accuracy"] - base_accuracy}"
)

print(
    f"Best action :"
    f"{random_result["best_action"]}"
)

print(
    f"Time"
    f"{random_time: .2f}"
)


Best accuracy : 0.3200
Improvement :-0.06
Best action :[2]
Time 2590.11


In [18]:
print("\nRandom Search trials: ")
for result in random_result["trials"]:
    print(
        f"Trial {result["trial"]:02d} |"
        f"Accuracy: {result["accuracy"]: .4f}"
        f"Actions: {result["actions"]}"
    )
    


Random Search trials: 
Trial 00 |Accuracy:  0.2200Actions: [0]
Trial 01 |Accuracy:  0.2600Actions: [3, 1, 2]
Trial 02 |Accuracy:  0.2200Actions: [1]
Trial 03 |Accuracy:  0.1400Actions: [6, 0, 1, 2, 4]
Trial 04 |Accuracy:  0.1600Actions: [0, 5]
Trial 05 |Accuracy:  0.2800Actions: [6, 1]
Trial 06 |Accuracy:  0.2200Actions: [4, 7, 0, 2]
Trial 07 |Accuracy:  0.1000Actions: [5, 2, 1, 3]
Trial 08 |Accuracy:  0.2200Actions: [1, 0, 5]
Trial 09 |Accuracy:  0.2800Actions: [5]
Trial 10 |Accuracy:  0.1800Actions: [4, 7, 0]
Trial 11 |Accuracy:  0.1400Actions: [1, 4, 0, 7]
Trial 12 |Accuracy:  0.0400Actions: [5, 4, 1]
Trial 13 |Accuracy:  0.2200Actions: [0]
Trial 14 |Accuracy:  0.2000Actions: [4, 0]
Trial 15 |Accuracy:  0.1800Actions: [1, 4]
Trial 16 |Accuracy:  0.2400Actions: [7, 5, 2]
Trial 17 |Accuracy:  0.2000Actions: [5, 2]
Trial 18 |Accuracy:  0.1600Actions: [4, 6]
Trial 19 |Accuracy:  0.3200Actions: [2]


In [19]:
print("\n" + "=" * 60)
print("BASELINE SUMMARY")
print("=" * 60)

print(
    f"{'Method':<20}"
    f"{'Accuracy':<15}"
    f"{'Improvement':<15}"
)

print("-" * 50)

print(
    f"{'Base prompt':<20}"
    f"{base_accuracy:<15.4f}"
    f"{0.0:<+15.4f}"
)

print(
    f"{'Manual':<20}"
    f"{manual_result['best_accuracy']:<15.4f}"
    f"{manual_result['best_accuracy'] - base_accuracy:<+15.4f}"
)

print(
    f"{'Random Search':<20}"
    f"{random_result['best_accuracy']:<15.4f}"
    f"{random_result['best_accuracy'] - base_accuracy:<+15.4f}"
)



BASELINE SUMMARY
Method              Accuracy       Improvement    
--------------------------------------------------
Base prompt         0.3800         +0.0000        
Manual              0.4400         +0.0600        
Random Search       0.3200         -0.0600        


In [20]:
import json

chemin_fichier = "/kaggle/working/baseline_results.json"

results = {

    "configuration": {
        "dataset_size": DATASET_SIZE,
        "model": MODEL_NAME,
        "max_steps": MAX_STEPS,
        "random_search_trials": RANDOM_SEARCH_TRIALS,
        "random_seed": RANDOM_SEED,
    },


    "manual": {
        "best_accuracy": manual_result["best_accuracy"],
        "best_actions": manual_result["best_actions"],
        "best_prompt": manual_result["best_prompt"],
        "time": manual_time,
        "results": manual_result["results"],
    },

    "random_search": {
        "best_accuracy": random_result["best_accuracy"],
        "best_action": random_result["best_action"],
        "best_prompt": random_result["best_prompt"],
        "time": random_time,
        "trials": random_result["trials"],
    },
}

with open(chemin_fichier, "w", encoding="utf-8") as file:
    json.dump(results, file, indent=4, ensure_ascii=False)

print(f"\nResults saved to {chemin_fichier}")



Results saved to /kaggle/working/baseline_results.json


In [21]:
# display the json file 
with open("/kaggle/working/baseline_results.json", "r", encoding="utf-8") as file:
    print(file.read())


{
    "configuration": {
        "dataset_size": 50,
        "model": "Qwen/Qwen2.5-3B-Instruct",
        "max_steps": 5,
        "random_search_trials": 20,
        "random_seed": 42
    },
    "manual": {
        "best_accuracy": 0.44,
        "best_actions": [
            3,
            7
        ],
        "best_prompt": "Solve the following math problem.\nProvide the final answer clearly.\n\nCheck your calculations carefully.\n\nClearly state the final answer at the end.",
        "time": 1297.1477315425873,
        "results": [
            {
                "actions": [
                    0
                ],
                "prompt": "Solve the following math problem.\nProvide the final answer clearly.\n\nSolve the problem step by step.",
                "accuracy": 0.22
            },
            {
                "actions": [
                    3
                ],
                "prompt": "Solve the following math problem.\nProvide the final answer clearly.\n\nCheck your c